# NB03 — Circuit Visualization

Three experiments that go beyond numbers and make the circuit structure directly visible:

1. **Per-layer similarity on z-neighbors** — for cross-class pairs that nb02 groups together, are they actually similar at the activation level, and at which layers? Validates that z-space groupings reflect real circuit sharing, not noise.
2. **Layer trajectory UMAP** — each image as a *path* through a shared 2D projection, showing how its representation evolves from layer 1 (generic) to layer 8 (class-specific). Compares nb01 vs nb02.
3. **Image-thumbnail UMAP + region inspector** — place actual CIFAR images at their z-space positions. See directly which images are near each other, and inspect regions to understand what the model is grouping.

**Dependency:** Requires nb02 checkpoints (nb02_phase1.pt, nb02_best.pt) and nb01 checkpoint (best.pt).

In [ ]:
import subprocess, sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DRIVE_ROOT = '/content/drive/MyDrive/ctls'

if IN_COLAB:
    REPO = '/content/trainable-circuits'
    if not os.path.exists(REPO):
        DRIVE_REPO = f'{DRIVE_ROOT}/trainable-circuits'
        if os.path.exists(DRIVE_REPO):
            import shutil
            shutil.copytree(DRIVE_REPO, REPO)
        else:
            subprocess.run(['git', 'clone', 'https://github.com/jacobposchl/trainable-circuits', REPO], check=True)
    sys.path.insert(0, REPO)
else:
    REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
    sys.path.insert(0, REPO)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'umap-learn'], check=False)
print(f'Repo: {REPO}')

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
import torchvision.transforms as transforms
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.lines import Line2D
import matplotlib.cm as cm
from sklearn.cluster import KMeans
from tqdm.auto import tqdm
import umap
import warnings
warnings.filterwarnings('ignore')

from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from models.soft_mask import SoftMask

print('Imports OK')

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = '/content/data' if IN_COLAB else os.path.join(REPO, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

# ── Checkpoint paths ──────────────────────────────────────────────────────────
# Set CKPT_DIR to the folder that contains 'unified_a/' and 'step5/' subfolders.
# From your Drive layout: /content/drive/MyDrive/ctls/experiments/
if IN_COLAB:
    CKPT_DIR = '/content/drive/MyDrive/ctls/experiments'
else:
    CKPT_DIR = os.path.join(REPO, 'checkpoints')

NB01_BEST = os.path.join(CKPT_DIR, 'unified_a', 'best.pt')
NB02_BEST = os.path.join(CKPT_DIR, 'step5',     'best.pt')

CIFAR_MEAN    = (0.4914, 0.4822, 0.4465)
CIFAR_STD     = (0.2470, 0.2435, 0.2616)
CIFAR_CLASSES = ['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']
N_CLASSES     = 10
CMAP          = plt.get_cmap('tab10')
CLASS_COLORS  = [CMAP(i) for i in range(N_CLASSES)]

val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

print(f'Device:    {DEVICE}')
print(f'nb01 ckpt: {NB01_BEST}')
print(f'nb02 ckpt: {NB02_BEST}')

# Sanity check — fail fast if paths are wrong before wasting GPU time
for p in [NB01_BEST, NB02_BEST]:
    if not os.path.exists(p):
        print(f'WARNING: checkpoint not found: {p}')
    else:
        size_mb = os.path.getsize(p) / 1e6
        print(f'  OK  {os.path.basename(os.path.dirname(p))}/best.pt  ({size_mb:.1f} MB)')

In [ ]:
def build_model(device):
    soft_mask = SoftMask(init_temperature=1.0)
    backbone  = CTLSBackbone(arch='resnet18', num_classes=10, pretrained=False,
                              soft_mask=soft_mask).to(device)
    meta_enc  = MetaEncoder(
        layer_dims=backbone.layer_dims, embedding_dim=64,
        encoder_type='weighted_sum', projection_dim=128,
    ).to(device)
    return backbone, meta_enc


def load_weights(path, backbone, meta_enc, device):
    """Load checkpoint regardless of whether it was saved by UnifiedTrainer or the notebook."""
    ckpt = torch.load(path, map_location=device)

    backbone_key = None
    for k in ('backbone', 'backbone_state', 'backbone_state_dict', 'model_state_dict', 'state_dict'):
        if k in ckpt:
            backbone_key = k
            break

    meta_key = None
    for k in ('meta_enc', 'meta_encoder', 'meta_encoder_state', 'meta_enc_state_dict', 'meta_encoder_state_dict'):
        if k in ckpt:
            meta_key = k
            break

    if backbone_key is None or meta_key is None:
        print(f'  Checkpoint keys: {list(ckpt.keys())}')
        raise KeyError(
            f'Could not find backbone/meta_enc keys. Available: {list(ckpt.keys())}'
        )

    backbone.load_state_dict(ckpt[backbone_key])
    meta_enc.load_state_dict(ckpt[meta_key])
    acc = ckpt.get('val_acc', ckpt.get('best_acc', float('nan')))
    print(f'  Loaded {os.path.relpath(path)}  '
          f'(keys: {backbone_key!r}, {meta_key!r})  val_acc={acc:.4f}')


print('Loading nb01...')
backbone_nb01, meta_enc_nb01 = build_model(DEVICE)
load_weights(NB01_BEST, backbone_nb01, meta_enc_nb01, DEVICE)

print('Loading nb02...')
backbone_nb02, meta_enc_nb02 = build_model(DEVICE)
load_weights(NB02_BEST, backbone_nb02, meta_enc_nb02, DEVICE)

backbone_nb01.eval(); meta_enc_nb01.eval()
backbone_nb02.eval(); meta_enc_nb02.eval()
print('Models ready.')

In [ ]:
@torch.no_grad()
def extract_all(backbone, meta_enc, root, device, n=400, seed=42):
    """Extract everything needed for all three visualization experiments.
    
    Returns:
      traj_all  : list[L] of ndarray [N, C_l]  — raw per-layer activations
      proj_all  : list[L] of ndarray [N, 128]  — per-layer projected embeddings
      z_all     : ndarray [N, 64]              — z embeddings (L2-normalised)
      imgs_all  : ndarray [N, 32, 32, 3] uint8 — raw images for display
      lbls_all  : ndarray [N]                  — class labels
    """
    ds_raw  = CIFAR10(root=root, train=False, transform=None,   download=True)
    ds_norm = CIFAR10(root=root, train=False, transform=val_tf, download=True)
    rng     = np.random.default_rng(seed)
    indices = rng.choice(len(ds_raw), n, replace=False)

    n_layers  = None
    traj_bufs = None
    proj_bufs = None
    z_buf, imgs_buf, lbls_buf = [], [], []

    for idx in tqdm(indices, desc='Extracting', leave=False):
        x  = ds_norm[idx][0].unsqueeze(0).to(device)
        _, traj = backbone(x)
        z  = meta_enc(traj)          # [1, 64]

        if n_layers is None:
            n_layers  = len(traj)
            traj_bufs = [[] for _ in range(n_layers)]
            proj_bufs = [[] for _ in range(n_layers)]

        for l, h in enumerate(traj):
            h_flat = h.view(h.shape[0], -1)          # [1, C_l] after avgpool
            traj_bufs[l].append(h_flat.squeeze(0).cpu().numpy())
            p = F.normalize(meta_enc.projectors[l](h_flat.squeeze(0)), dim=-1)
            proj_bufs[l].append(p.cpu().numpy())

        z_buf.append(F.normalize(z, dim=-1).squeeze(0).cpu().numpy())
        imgs_buf.append(np.array(ds_raw[idx][0]))
        lbls_buf.append(ds_raw[idx][1])

    traj_all = [np.stack(t) for t in traj_bufs]   # list of [N, C_l]
    proj_all = [np.stack(p) for p in proj_bufs]   # list of [N, 128]
    z_all    = np.stack(z_buf)                     # [N, 64]
    imgs_all = np.stack(imgs_buf)                  # [N, 32, 32, 3]
    lbls_all = np.array(lbls_buf)                  # [N]
    print(f'Extracted {n} images, {n_layers} layers')
    return traj_all, proj_all, z_all, imgs_all, lbls_all


print('Extracting nb01...')
traj_nb01, proj_nb01, z_nb01, imgs, lbls = extract_all(
    backbone_nb01, meta_enc_nb01, DATA_DIR, DEVICE, n=400)

print('Extracting nb02...')
traj_nb02, proj_nb02, z_nb02, _, _ = extract_all(
    backbone_nb02, meta_enc_nb02, DATA_DIR, DEVICE, n=400)

N_LAYERS = len(traj_nb01)
N        = len(lbls)
print(f'N={N}, L={N_LAYERS}')

---
## Section 2 — Per-Layer Similarity on Z-Neighbors

**Question:** When nb02 groups a horse and a cat together in z-space, are their backbone activations
actually similar at earlier layers? If yes, the grouping reflects genuine low-level circuit sharing.
If similarity only appears at layer 8, it's just visual confusability.

**Method:** Find the top cross-class z-neighbors from nb02. Compute per-layer cosine similarity
for three groups:
- **Cross-class z-neighbors** — different class, high z-similarity (the interesting pairs)
- **Same-class z-neighbors** — same class, high z-similarity (the baseline)
- **Random cross-class** — different class, random (the null hypothesis)

If cross-class z-neighbors show elevated similarity at *early* layers (1–4), that's the finding.

In [ ]:
def find_pairs(z, lbls, n_pairs=80, mode='cross_neighbor', rng_seed=0):
    """Find pairs of images.
    mode: 'cross_neighbor'  — different class, highest z-similarity
          'same_neighbor'   — same class, highest z-similarity
          'random_cross'    — different class, random (not top-k)
    Returns list of (i, j) tuples.
    """
    z_norm = z / (np.linalg.norm(z, axis=1, keepdims=True) + 1e-8)
    sim    = z_norm @ z_norm.T   # [N, N]
    np.fill_diagonal(sim, -2.0)
    rng    = np.random.default_rng(rng_seed)
    pairs  = []

    if mode == 'cross_neighbor':
        # For each image, find its most similar image from a DIFFERENT class
        for i in range(len(lbls)):
            mask = lbls != lbls[i]
            if not mask.any(): continue
            row  = sim[i].copy(); row[~mask] = -2.0
            j    = int(np.argmax(row))
            pairs.append((i, j))
        # Sort by z-similarity descending, take top n_pairs
        pairs.sort(key=lambda p: sim[p[0], p[1]], reverse=True)
        pairs = pairs[:n_pairs]

    elif mode == 'same_neighbor':
        for i in range(len(lbls)):
            mask = lbls == lbls[i]
            mask[i] = False
            if not mask.any(): continue
            row  = sim[i].copy(); row[~mask] = -2.0
            j    = int(np.argmax(row))
            pairs.append((i, j))
        pairs.sort(key=lambda p: sim[p[0], p[1]], reverse=True)
        pairs = pairs[:n_pairs]

    elif mode == 'random_cross':
        # Random cross-class pairs, excluding the top-20 z-neighbors
        attempts = 0
        while len(pairs) < n_pairs and attempts < 10000:
            i, j = rng.integers(0, len(lbls), 2)
            if i == j or lbls[i] == lbls[j]: 
                attempts += 1; continue
            # Exclude high-similarity pairs (those would be z-neighbors)
            if sim[i, j] > np.percentile(sim[sim > -2], 70):
                attempts += 1; continue
            pairs.append((i, j))
            attempts += 1

    return pairs


cross_pairs  = find_pairs(z_nb02, lbls, n_pairs=80, mode='cross_neighbor')
same_pairs   = find_pairs(z_nb02, lbls, n_pairs=80, mode='same_neighbor')
random_pairs = find_pairs(z_nb02, lbls, n_pairs=80, mode='random_cross')

print(f'Cross-class z-neighbors: {len(cross_pairs)} pairs')
print(f'Same-class z-neighbors:  {len(same_pairs)} pairs')
print(f'Random cross-class:       {len(random_pairs)} pairs')

# Show class pair breakdown for cross-class neighbors
from collections import Counter
pair_classes = Counter()
for i, j in cross_pairs:
    a, b = sorted([CIFAR_CLASSES[lbls[i]], CIFAR_CLASSES[lbls[j]]])
    pair_classes[(a, b)] += 1
print('\nTop cross-class pairs by frequency:')
for (a, b), cnt in pair_classes.most_common(8):
    print(f'  {a:8s} ↔ {b:8s} : {cnt}')

In [ ]:
def per_layer_sims(traj, pairs):
    """Per-layer cosine similarity for a list of (i,j) pairs.
    Returns [n_pairs, n_layers] float array.
    """
    results = []
    for i, j in pairs:
        row = []
        for l in range(len(traj)):
            h_a = traj[l][i]   # [C_l]
            h_b = traj[l][j]
            n_a = np.linalg.norm(h_a) + 1e-8
            n_b = np.linalg.norm(h_b) + 1e-8
            row.append(float(np.dot(h_a, h_b) / (n_a * n_b)))
        results.append(row)
    return np.array(results, dtype=np.float32)  # [P, L]


sim_cross  = per_layer_sims(traj_nb02, cross_pairs)   # [P, L]
sim_same   = per_layer_sims(traj_nb02, same_pairs)
sim_random = per_layer_sims(traj_nb02, random_pairs)

# Also compute on nb01 cross-class pairs for comparison
cross_pairs_nb01 = find_pairs(z_nb01, lbls, n_pairs=80, mode='cross_neighbor')
sim_cross_nb01   = per_layer_sims(traj_nb01, cross_pairs_nb01)

layer_ids = [f'h{l+1}' for l in range(N_LAYERS)]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (title, groups) in zip(axes, [
    ('nb02: per-layer cosine sim by pair type',
     [('cross-class z-nbrs', sim_cross,  'royalblue', '-'),
      ('same-class z-nbrs',  sim_same,   'forestgreen', '--'),
      ('random cross-class', sim_random, 'tomato', ':')]),
    ('Cross-class z-neighbors: nb01 vs nb02',
     [('nb02 cross-class z-nbrs', sim_cross,     'royalblue', '-'),
      ('nb01 cross-class z-nbrs', sim_cross_nb01,'darkorange',  '--')]),
]):
    for label, sim_mat, color, ls in groups:
        mu    = sim_mat.mean(0)
        se    = sim_mat.std(0) / np.sqrt(len(sim_mat))
        xs    = np.arange(N_LAYERS)
        ax.plot(xs, mu, color=color, linestyle=ls, linewidth=2, marker='o',
                markersize=5, label=label)
        ax.fill_between(xs, mu - se, mu + se, alpha=0.15, color=color)
    ax.set_xticks(np.arange(N_LAYERS)); ax.set_xticklabels(layer_ids)
    ax.set_xlabel('Layer'); ax.set_ylabel('Mean cosine similarity')
    ax.set_title(title); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('Per-layer cosine similarity on z-space neighbor pairs', fontsize=12)
plt.tight_layout(); plt.show()

# Print early-layer summary
print('\nEarly-layer (h1–h4) mean similarity:')
for label, sim_mat in [('cross-class z-nbrs', sim_cross),
                        ('same-class z-nbrs',  sim_same),
                        ('random cross-class', sim_random)]:
    early = sim_mat[:, :4].mean()
    late  = sim_mat[:, 4:].mean()
    print(f'  {label:25s}: early={early:.3f}  late={late:.3f}')

In [ ]:
# Statistical test: at each layer, is cross-class z-neighbor sim > random cross-class?
print('Mann-Whitney U test: cross-class z-neighbors vs random cross-class, per layer')
print(f'  {"Layer":6s} | {"cross μ":8s} | {"random μ":9s} | {"U p-value":10s} | significant?')
print('  ' + '-'*60)
for l in range(N_LAYERS):
    u_stat, p_val = stats.mannwhitneyu(
        sim_cross[:, l], sim_random[:, l], alternative='greater')
    sig = '✓' if p_val < 0.05 else ' '
    print(f'  h{l+1:1d}     | {sim_cross[:,l].mean():8.3f} | {sim_random[:,l].mean():9.3f} | {p_val:10.4f} | {sig}')

print()
print('Interpretation:')
print('  If early layers (h1-h4) are significant: cross-class z-neighbors share')
print('  genuine low-level circuits, not just semantic confusability.')
print('  If only h7-h8 are significant: z is grouping visually confusable images')
print('  but not discovering deep circuit sharing.')

In [ ]:
# ── Cross-class z-neighbor pair gallery ──────────────────────────────────────
# Shows the top-N pairs the model considers circuit-similar despite different class labels
z_nb02_norm = z_nb02 / (np.linalg.norm(z_nb02, axis=1, keepdims=True) + 1e-8)
z_sim_mat   = z_nb02_norm @ z_nb02_norm.T

N_SHOW    = 8   # top pairs by z-space cosine similarity
top_pairs = sorted(cross_pairs, key=lambda p: z_sim_mat[p[0], p[1]], reverse=True)[:N_SHOW]

# Layout: 2 pairs per row  =>  4 image columns (anchor, neighbor, anchor, neighbor)
n_rows = (N_SHOW + 1) // 2
n_cols = 4
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 5 * n_rows))
axes = np.array(axes).reshape(n_rows, n_cols)

for k, (i, j) in enumerate(top_pairs):
    row, col_off = divmod(k, 2)
    col_off *= 2
    z_sim = float(z_sim_mat[i, j])

    ax_a = axes[row, col_off]
    ax_a.imshow(imgs[i])
    ax_a.set_title(f'ANCHOR\n{CIFAR_CLASSES[lbls[i]]}', fontsize=12,
                   fontweight='bold', color='darkred', pad=6)
    ax_a.axis('off')
    for sp in ax_a.spines.values():
        sp.set_visible(True); sp.set_edgecolor(CLASS_COLORS[lbls[i]]); sp.set_linewidth(4)

    ax_n = axes[row, col_off + 1]
    ax_n.imshow(imgs[j])
    ax_n.set_title(f'Z-NEIGHBOR  (z-sim={z_sim:.3f})\n{CIFAR_CLASSES[lbls[j]]}',
                   fontsize=12, fontweight='bold', color='navy', pad=6)
    ax_n.axis('off')
    for sp in ax_n.spines.values():
        sp.set_visible(True); sp.set_edgecolor(CLASS_COLORS[lbls[j]]); sp.set_linewidth(4)

fig.suptitle(
    'Top cross-class z-neighbors (nb02)\n'
    'Images the model considers circuit-similar despite different class labels',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

print(f'\nTop {N_SHOW} cross-class z-neighbor pairs (by z-space cosine similarity):')
for k, (i, j) in enumerate(top_pairs):
    print(f'  {k+1}. {CIFAR_CLASSES[lbls[i]]:8s} <-> {CIFAR_CLASSES[lbls[j]]:8s}  z-sim={z_sim_mat[i,j]:.3f}')


---
## Section 4 — Per-Layer UMAP Gallery

8 mini-UMAPs side by side, one for each layer's projected embeddings. Shows when class
structure emerges as you go deeper.

**nb01** should show: no structure at layers 1–4, then rapid class separation at 7–8.  
**nb02** should show: class structure emerging more gradually, or more diffuse at late layers.

In [ ]:
# Fit a UMAP for each layer independently
def fit_per_layer_umaps(proj_all, n_neighbors=15, min_dist=0.3, seed=42):
    umaps = []
    for l, proj in enumerate(proj_all):
        r = umap.UMAP(n_components=2, n_neighbors=n_neighbors,
                      min_dist=min_dist, metric='cosine', random_state=seed)
        umaps.append(r.fit_transform(proj))
    return umaps

print('Fitting per-layer UMAPs for nb01...')
layer_umaps_nb01 = fit_per_layer_umaps(proj_nb01)
print('Fitting per-layer UMAPs for nb02...')
layer_umaps_nb02 = fit_per_layer_umaps(proj_nb02)

# Plot: 2 rows (nb01, nb02) × L columns
fig, axes = plt.subplots(2, N_LAYERS, figsize=(2.8*N_LAYERS, 6))
for row, (umaps, model) in enumerate([(layer_umaps_nb01, 'nb01'),
                                       (layer_umaps_nb02, 'nb02')]):
    for l, (u2d, ax) in enumerate(zip(umaps, axes[row])):
        for c in range(N_CLASSES):
            mask = lbls == c
            ax.scatter(u2d[mask, 0], u2d[mask, 1], s=8,
                       color=CLASS_COLORS[c], alpha=0.6, label=CIFAR_CLASSES[c])
        ax.set_title(f'{model}  h{l+1}', fontsize=8)
        ax.axis('off')

# Shared class legend
handles = [mpatches.Patch(color=CLASS_COLORS[c], label=CIFAR_CLASSES[c])
           for c in range(N_CLASSES)]
fig.legend(handles=handles, fontsize=7, ncol=5,
           loc='lower center', bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Per-layer projected embedding UMAP (row 1 = nb01, row 2 = nb02)',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.06, 1, 1]); plt.show()

---
## Section 5 — Coordinate-Based UMAP Inspector

**How to use:**
1. Run `nb03_umap_overview` to see the UMAP with 0–100 coordinates on both axes.
2. Read off a coordinate of interest (e.g. a dense cluster, an isolated region, a boundary).
3. Edit `QUERY_X`, `QUERY_Y`, `K_NEAREST` in `nb03_umap_query` and run it.
4. You get: (a) a full-scale image grid of the K nearest images, and (b) a zoomed UMAP region showing exactly where each retrieved image sits.


In [ ]:
# ── Fit independent UMAPs for nb01 and nb02 z-space ─────────────────────────
def fit_umap_scaled(z, seed=42):
    """Fit UMAP and scale coordinates to [0, 100] x [0, 100]."""
    z_norm  = z / (np.linalg.norm(z, axis=1, keepdims=True) + 1e-8)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.3, random_state=seed)
    raw     = reducer.fit_transform(z_norm)
    mn, mx  = raw.min(axis=0), raw.max(axis=0)
    scaled  = (raw - mn) / (mx - mn) * 100
    return scaled

print('Fitting UMAPs (this takes ~20s)...')
umap_nb01 = fit_umap_scaled(z_nb01)
umap_nb02 = fit_umap_scaled(z_nb02)
print('Done. Both UMAPs scaled to [0, 100] x [0, 100].')

# ── Overview plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, coords, title in [
    (axes[0], umap_nb01, 'nb01 z-space UMAP'),
    (axes[1], umap_nb02, 'nb02 z-space UMAP  <-- copy a coordinate from here'),
]:
    for c in range(N_CLASSES):
        mask = lbls == c
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   s=20, color=CLASS_COLORS[c], alpha=0.75, label=CIFAR_CLASSES[c])
    ax.set_xlim(-2, 102); ax.set_ylim(-2, 102)
    ax.set_xticks(range(0, 101, 10))
    ax.set_yticks(range(0, 101, 10))
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('x  (0-100)', fontsize=12)
    ax.set_ylabel('y  (0-100)', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')

axes[1].legend(loc='upper right', fontsize=9, markerscale=1.5, framealpha=0.9, ncol=2)
fig.suptitle(
    'z-space UMAP overview  |  Read a coordinate then run the query cell below',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()


In [ ]:
# ======================================================================
# EDIT THESE LINES, then run the cell
# ======================================================================
QUERY_X   = 50      # x coordinate from the UMAP overview (0-100)
QUERY_Y   = 50      # y coordinate from the UMAP overview (0-100)
K_NEAREST = 8       # number of nearest images to retrieve
WHICH     = 'nb02'  # 'nb01' or 'nb02'
# ======================================================================

coords = umap_nb02 if WHICH == 'nb02' else umap_nb01
query  = np.array([QUERY_X, QUERY_Y], dtype=float)
dists  = np.linalg.norm(coords - query, axis=1)
nn_idx = np.argsort(dists)[:K_NEAREST]

print(f'\n{WHICH} -- query ({QUERY_X}, {QUERY_Y})  ->  {K_NEAREST} nearest images\n')
print(f'  {"rank":>4}  {"idx":>4}  {"class":<8}  {"coord":>14}  {"dist":>6}')
print('  ' + '-' * 46)
for rank, idx in enumerate(nn_idx):
    cx, cy = coords[idx]
    print(f'  {rank+1:>4}  {idx:>4}  {CIFAR_CLASSES[lbls[idx]]:<8}  ({cx:5.1f}, {cy:5.1f})  {dists[idx]:6.2f}')

# ── Plot 1: Full-scale image grid ─────────────────────────────────────────────
ncols = min(4, K_NEAREST)
nrows = (K_NEAREST + ncols - 1) // ncols
fig1, axes1 = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 5.2 * nrows))
axes1 = np.array(axes1).reshape(-1)

for k, idx in enumerate(nn_idx):
    ax = axes1[k]
    ax.imshow(imgs[idx])
    cx, cy = coords[idx]
    ax.set_title(
        f'#{k+1}  {CIFAR_CLASSES[lbls[idx]]}\n'
        f'coord ({cx:.1f}, {cy:.1f})  |  dist {dists[idx]:.2f}',
        fontsize=11, fontweight='bold', pad=6
    )
    for sp in ax.spines.values():
        sp.set_visible(True)
        sp.set_edgecolor(CLASS_COLORS[lbls[idx]])
        sp.set_linewidth(4)
    ax.set_xticks([]); ax.set_yticks([])

for k in range(K_NEAREST, len(axes1)):
    axes1[k].axis('off')

fig1.suptitle(
    f'{WHICH}  |  {K_NEAREST} nearest images to ({QUERY_X}, {QUERY_Y})',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

# ── Plot 2: Zoomed UMAP region around query point ─────────────────────────────
PAD  = 15   # zoom radius in coordinate units
x0   = max(0.0,   QUERY_X - PAD)
x1_  = min(100.0, QUERY_X + PAD)
y0   = max(0.0,   QUERY_Y - PAD)
y1_  = min(100.0, QUERY_Y + PAD)

in_view = set(np.where(
    (coords[:, 0] >= x0) & (coords[:, 0] <= x1_) &
    (coords[:, 1] >= y0) & (coords[:, 1] <= y1_)
)[0].tolist())

fig2, ax2 = plt.subplots(figsize=(8, 8))

# Background points in view (small, dim)
for c in range(N_CLASSES):
    iv_c = [idx2 for idx2 in in_view if lbls[idx2] == c]
    if iv_c:
        ax2.scatter(coords[iv_c, 0], coords[iv_c, 1],
                    s=35, color=CLASS_COLORS[c], alpha=0.3, zorder=2)

# Retrieved K images (larger, opaque, black edge, labeled)
nn_set = set(nn_idx.tolist())
for rank, idx in enumerate(nn_idx):
    cx, cy = coords[idx]
    ax2.scatter(cx, cy, s=280, color=CLASS_COLORS[lbls[idx]],
                edgecolors='black', linewidths=2, zorder=4)
    ax2.annotate(
        f'#{rank+1} {CIFAR_CLASSES[lbls[idx]]}',
        (cx, cy), xytext=(6, 6), textcoords='offset points',
        fontsize=9, fontweight='bold', zorder=5
    )

# Query point star
ax2.scatter([QUERY_X], [QUERY_Y], s=400, marker='*',
            color='black', zorder=6, label='query point')

# Legend
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
handles = [mpatches.Patch(color=CLASS_COLORS[c], label=CIFAR_CLASSES[c])
           for c in range(N_CLASSES)]
handles.append(Line2D([0], [0], marker='*', color='black', markersize=14,
                      linestyle='None', label='query'))
ax2.legend(handles=handles, fontsize=8, loc='upper right', ncol=2, framealpha=0.9)

ax2.set_xlim(x0, x1_); ax2.set_ylim(y0, y1_)
step = 5
ax2.set_xticks(np.arange(int(x0 // step) * step, x1_ + 1, step))
ax2.set_yticks(np.arange(int(y0 // step) * step, y1_ + 1, step))
ax2.grid(True, alpha=0.4)
ax2.set_xlabel('UMAP x', fontsize=12)
ax2.set_ylabel('UMAP y', fontsize=12)
ax2.set_title(
    f'Zoomed region around ({QUERY_X}, {QUERY_Y})  +/-{PAD} units\n'
    f'{WHICH} z-space  |  labeled = retrieved images',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()
